# 04 — Phân lớp có giám sát (họ bài toán phụ) & So sánh hai họ mô hình · Đồ án 6
**Phần E (Mong).** Câu hỏi nghiệp vụ: *với ngân sách rà soát 100 giao dịch mỗi ngày, mô hình phân lớp có giám sát bắt được bao nhiêu giao dịch gian lận so với baseline và so với phát hiện bất thường không giám sát?*

**Trạng thái giao thức (Mục E1 — Đã đồng bộ với Cường B, Quỳnh C, Huy D):**
1. **Nguồn dữ liệu & Khóa (`D-01`, `D-02`):** Đọc từ `src/preprocessing.py` (`data/processed/transactions_processed.csv`), khóa chính `txn_id` (`1..284807`), chỉ đưa vào mô hình các dòng `in_model_set == 1` (`283.726` dòng, `473` gian lận; đã loại `1.081` bản sao thừa `DQ-01`).
2. **Đặc trưng & Tiền xử lý (`D-04`, `D-07`):** `FEATURES = V1..V28 + amount_log1p` (29 cột). Bỏ `Time` thô. Dùng `make_preprocessor()` của Cường (`StandardScaler` chỉ fit trên `amount_log1p` của tập train; `V1..V28` giữ nguyên).
3. **Hai giao thức chia tập (`D-06`):**
   - **Giao thức chính theo ngày (`split_temporal`):** `train` = 80% Ngày 0 phân tầng (`115.388` dòng, `218` gian lận); `val` = 20% Ngày 0 phân tầng (`28.848` dòng, `54` gian lận, ngân sách tỉ lệ `K_VAL = 20`); `test` = Ngày 1 sau khử trùng (`139.490` dòng, `201` gian lận, ngân sách `K = 100` giao dịch/ngày, trần lý thuyết `Recall@100 = 100/201 = 49,75%`).
   - **Giao thức phụ đối sánh trực tiếp với Huy (`split_random`):** Chia 60/20/20 phân tầng (`train`: `170.235`, `val`: `56.745`, `test`: `56.746` dòng / `95` gian lận, `K = 100`) — khớp 1-1 theo `txn_id` với `outputs/anomaly_scores_test.csv` của Huy (Phần D).
4. **Chỉ số đánh giá:** `Precision@100`, `Recall@100`, `PR-AUC` (Average Precision). Không dùng `accuracy` làm chỉ số chính.

## 1. Cấu hình và nạp dữ liệu đã tiền xử lý

In [1]:
import hashlib, json, sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, accuracy_score

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "preprocessing.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.preprocessing import (
    EXPECTED_SHA256, FEATURES, LABEL, V_COLS, RANDOM_STATE,
    REVIEW_BUDGET_PER_DAY, file_sha256, load_raw, build_processed,
    load_processed, make_preprocessor
)

K = REVIEW_BUDGET_PER_DAY    # 100 giao dịch/ngày
VAL_FRACTION = 0.20          # split_temporal của Cường dùng 20% Ngày 0 làm validation
K_VAL = round(K * VAL_FRACTION)  # ngân sách tỉ lệ trên validation temporal = 20

RAW_PATH = ROOT / "data" / "raw" / "creditcard.csv"
PROC_PATH = ROOT / "data" / "processed" / "transactions_processed.csv"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

assert file_sha256(RAW_PATH) == EXPECTED_SHA256, "sha256 của creditcard.csv không khớp!"
if not PROC_PATH.exists():
    PROC_PATH.parent.mkdir(parents=True, exist_ok=True)
    build_processed(load_raw(RAW_PATH, verify=True)).to_csv(PROC_PATH, index=False, lineterminator="\n")

df = load_processed(PROC_PATH)
print(f"Đã nạp: {PROC_PATH.name} | shape={df.shape} | tổng fraud={int(df[LABEL].sum())} | in_model_set={int(df['in_model_set'].sum())} (fraud={int(df.loc[df['in_model_set']==1, LABEL].sum())})")

Đã nạp: transactions_processed.csv | shape=(284807, 47) | tổng fraud=492 | in_model_set=283726 (fraud=473)


## 2. Kiểm tra phân chia dữ liệu (`split_temporal` và `split_random`) & chống rò rỉ
Sử dụng trực tiếp cột `split_temporal` (chính) và `split_random` (phụ) trong `transactions_processed.csv` của Cường (Phần B).

In [2]:
train = df[df["split_temporal"] == "train"].copy()
val   = df[df["split_temporal"] == "val"].copy()
test  = df[df["split_temporal"] == "test"].copy()
day0  = pd.concat([train, val], axis=0).sort_values("txn_id")

print("=== GIAO THỨC CHÍNH: split_temporal (Học Ngày 0 -> Chấm Ngày 1) ===")
for name, part in [("train (80% ngày 0)", train), ("val (20% ngày 0)", val), ("test (ngày 1)", test)]:
    print(f"{name:20s} n={len(part):>7d}  fraud={int(part[LABEL].sum()):>4d} ({part[LABEL].mean()*100:.4f}%)")

# Kiểm tra không có bản sao hay rò rỉ giữa test (ngày 1) và ngày 0
assert int(day0["is_duplicate"].sum()) == 0 and int(test["is_duplicate"].sum()) == 0
dup_cols = V_COLS + ["amount", LABEL]
dup_cross = test[dup_cols].merge(day0[dup_cols].drop_duplicates(), on=dup_cols, how="inner")
print("Số dòng test trùng đặc trưng + nhãn với ngày 0:", len(dup_cross))
assert len(dup_cross) == 0

n_fraud_test = int(test[LABEL].sum())
ceiling_temporal = min(K, n_fraud_test) / n_fraud_test
print(f"Trần lý thuyết Recall@{K} ở Ngày 1 (sau khử trùng) = {K}/{n_fraud_test} = {ceiling_temporal:.4f} ({ceiling_temporal*100:.2f}%)")

print("\n=== GIAO THỨC PHỤ (Đối sánh 1-1 với Huy): split_random (60/20/20) ===")
for part_name in ["train", "val", "test"]:
    sub = df[df["split_random"] == part_name]
    print(f"{part_name:20s} n={len(sub):>7d}  fraud={int(sub[LABEL].sum()):>4d} ({sub[LABEL].mean()*100:.4f}%)")

=== GIAO THỨC CHÍNH: split_temporal (Học Ngày 0 -> Chấm Ngày 1) ===
train (80% ngày 0)   n= 115388  fraud= 218 (0.1889%)
val (20% ngày 0)     n=  28848  fraud=  54 (0.1872%)
test (ngày 1)        n= 139490  fraud= 201 (0.1441%)
Số dòng test trùng đặc trưng + nhãn với ngày 0: 0
Trần lý thuyết Recall@100 ở Ngày 1 (sau khử trùng) = 100/201 = 0.4975 (49.75%)

=== GIAO THỨC PHỤ (Đối sánh 1-1 với Huy): split_random (60/20/20) ===
train                n= 170235  fraud= 284 (0.1668%)
val                  n=  56745  fraud=  94 (0.1657%)
test                 n=  56746  fraud=  95 (0.1674%)


## 3. Hàm đánh giá

In [3]:
def precision_at_k(y, score, k=K):
    idx = np.argsort(-np.asarray(score), kind="stable")[:k]
    return float(np.asarray(y)[idx].mean())

def recall_at_k(y, score, k=K):
    y = np.asarray(y)
    idx = np.argsort(-np.asarray(score), kind="stable")[:k]
    return float(y[idx].sum() / y.sum())

def report(name, y, score, k=K):
    y = np.asarray(y)
    idx = np.argsort(-np.asarray(score), kind="stable")[:k]
    tp = int(y[idx].sum())
    return {
        "model": name,
        "PR_AUC": float(average_precision_score(y, score)),
        f"Precision@{k}": float(tp / k),
        f"Recall@{k}": float(tp / y.sum()),
        "TP": tp,
        "FP": int(k - tp),
        "FN": int(y.sum() - tp),
    }

## 4. Baseline (Mục E3)
- **Đoán lớp đa số:** luôn dự đoán `0` (không gian lận). Accuracy đạt `99,86%` trên Ngày 1 nhưng bắt được `0` gian lận -> minh chứng vì sao không dùng accuracy.
- **Xếp hạng ngẫu nhiên:** chạy 100 lần với seed cố định, lấy trung bình.

In [4]:
yv, yt = val[LABEL].to_numpy(), test[LABEL].to_numpy()

maj = np.zeros(len(yt), dtype=int)
print(f"Baseline đa số (Test Ngày 1): accuracy = {accuracy_score(yt, maj):.4f} ({accuracy_score(yt, maj)*100:.2f}%), gian lận bắt được = 0")

rng = np.random.default_rng(RANDOM_STATE)
def random_baseline(y, k, runs=100):
    res = [report("random", y, rng.random(len(y)), k) for _ in range(runs)]
    return pd.DataFrame(res).drop(columns="model").mean().to_dict()

rb_val = random_baseline(yv, K_VAL)
rb_test = random_baseline(yt, K)
print("Random baseline (Val temporal, K=20): ", {k: round(v, 5) for k, v in rb_val.items()})
print("Random baseline (Test Ngày 1, K=100):", {k: round(v, 5) for k, v in rb_test.items()})

Baseline đa số (Test Ngày 1): accuracy = 0.9986 (99.86%), gian lận bắt được = 0
Random baseline (Val temporal, K=20):  {'PR_AUC': 0.002, 'Precision@20': 0.0, 'Recall@20': 0.0, 'TP': 0.0, 'FP': 20.0, 'FN': 54.0}
Random baseline (Test Ngày 1, K=100): {'PR_AUC': 0.00154, 'Precision@100': 0.0015, 'Recall@100': 0.00075, 'TP': 0.15, 'FP': 99.85, 'FN': 200.85}


## 5. Huấn luyện và chọn tham số trên Validation (`split_temporal` — Mục E4, E5)
Ba họ mô hình (`LogisticRegression`, `RandomForestClassifier`, `HistGradientBoostingClassifier`). Xử lý mất cân bằng bằng `class_weight` chỉ trên tập train; `make_preprocessor(FEATURES)` chỉ fit `StandardScaler` cho `amount_log1p` trên train. Chọn cấu hình theo `PR_AUC` trên validation.

In [5]:
def make_model(name, **kw):
    if name == "logreg":
        clf = LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE, **kw)
    elif name == "random_forest":
        clf = RandomForestClassifier(class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE, **kw)
    elif name == "hist_gb":
        clf = HistGradientBoostingClassifier(class_weight="balanced", random_state=RANDOM_STATE, **kw)
    else:
        raise ValueError(name)
    return Pipeline([("prep", make_preprocessor(FEATURES)), ("clf", clf)])

grid = {
    "logreg": [{"C": 0.1}, {"C": 1.0}],
    "random_forest": [
        {"n_estimators": 200, "min_samples_leaf": 1},
        {"n_estimators": 200, "min_samples_leaf": 5},
    ],
    "hist_gb": [
        {"max_depth": 4, "learning_rate": 0.1},
        {"max_depth": 6, "learning_rate": 0.05},
    ],
}

X_tr, y_tr = train[FEATURES], train[LABEL].to_numpy()
X_val, y_val = val[FEATURES], val[LABEL].to_numpy()

val_rows = []
for name, cfgs in grid.items():
    for cfg in cfgs:
        m = make_model(name, **cfg).fit(X_tr, y_tr)
        s = m.predict_proba(X_val)[:, 1]
        r = report(name, y_val, s, K_VAL)
        r["config"] = json.dumps(cfg)
        val_rows.append(r)

val_table = pd.DataFrame(val_rows).sort_values("PR_AUC", ascending=False).reset_index(drop=True)
val_table[["model", "PR_AUC", f"Precision@{K_VAL}", f"Recall@{K_VAL}", "config"]]

,model,PR_AUC,Precision@20,Recall@20,config
0,random_forest,0.870316,0.95,0.351852,"{""n_estimators"": 200, ""min_samples_leaf"": 1}"
1,random_forest,0.852669,0.95,0.351852,"{""n_estimators"": 200, ""min_samples_leaf"": 5}"
2,hist_gb,0.847761,0.95,0.351852,"{""max_depth"": 4, ""learning_rate"": 0.1}"
3,hist_gb,0.739391,0.75,0.277778,"{""max_depth"": 6, ""learning_rate"": 0.05}"
4,logreg,0.684004,0.75,0.277778,"{""C"": 1.0}"
5,logreg,0.680943,0.75,0.277778,"{""C"": 0.1}"


## 6. Cấu hình tốt nhất trên Validation cho mỗi mô hình

In [6]:
best = (
    val_table.sort_values("PR_AUC", ascending=False)
    .groupby("model", as_index=False)
    .head(1)
    .reset_index(drop=True)
)
best[["model", "PR_AUC", f"Precision@{K_VAL}", f"Recall@{K_VAL}", "config"]]

,model,PR_AUC,Precision@20,Recall@20,config
0,random_forest,0.870316,0.95,0.351852,"{""n_estimators"": 200, ""min_samples_leaf"": 1}"
1,hist_gb,0.847761,0.95,0.351852,"{""max_depth"": 4, ""learning_rate"": 0.1}"
2,logreg,0.684004,0.75,0.277778,"{""C"": 1.0}"


## 7. Đánh giá cuối trên Test Ngày 1 (`split_temporal`, 139.490 dòng, 201 gian lận) — Chạy một lần (Mục E6)
Huấn luyện lại trên toàn bộ Ngày 0 (`train + val`: 144.236 dòng, 272 gian lận) với cấu hình đã chọn trên Validation, rồi chấm điểm Ngày 1.

In [7]:
X_d0, y_d0 = day0[FEATURES], day0[LABEL].to_numpy()
X_te, y_te = test[FEATURES], test[LABEL].to_numpy()

final_rows, test_scores = [], {}
for _, b in best.iterrows():
    name = b["model"]
    cfg = json.loads(b["config"])
    m = make_model(name, **cfg).fit(X_d0, y_d0)
    s = m.predict_proba(X_te)[:, 1]
    test_scores[name] = s
    final_rows.append(report(name, y_te, s, K))

final_rows.append({"model": "random_baseline_mean", **rb_test})
final = pd.DataFrame(final_rows)
final[["model", "PR_AUC", "Precision@100", "Recall@100", "TP", "FP", "FN"]]

,model,PR_AUC,Precision@100,Recall@100,TP,FP,FN
0,random_forest,0.784326,0.9300,0.462687,93.00,7.00,108.00
1,hist_gb,0.672652,0.9500,0.472637,95.00,5.00,106.00
2,logreg,0.723883,0.9400,0.467662,94.00,6.00,107.00
3,random_baseline_mean,0.001541,0.0015,0.000746,0.15,99.85,200.85


## 8. Phân tích lỗi trên Test Ngày 1 theo dải số tiền (`amount_band`)

In [8]:
top_model = final[final["model"] != "random_baseline_mean"].sort_values("PR_AUC", ascending=False).iloc[0]["model"]
s_top = test_scores[top_model]
order = np.argsort(-s_top, kind="stable")[:K]
flagged_ids = set(test.iloc[order]["txn_id"])

tp = int(test.iloc[order][LABEL].sum())
fp = K - tp
missed = test[(test[LABEL] == 1) & (~test["txn_id"].isin(flagged_ids))]

print(f"Mô hình có PR-AUC cao nhất: {top_model} | Trong Top {K}: bắt đúng {tp}, báo động giả {fp} | Bỏ sót {len(missed)}/{n_fraud_test}")
print("\nThống kê Amount của các giao dịch gian lận bị bỏ sót:")
print(missed["amount"].describe().round(2))

# Bảng phân rã theo dải số tiền (amount_band)
test_band = test[["txn_id", "amount_band_id", "amount_band", LABEL]].copy()
test_band["in_top100"] = test_band["txn_id"].isin(flagged_ids).astype(int)
band_summary = (
    test_band.groupby(["amount_band_id", "amount_band"], as_index=False)
    .agg(
        so_giao_dich=(LABEL, "count"),
        gian_lan=(LABEL, "sum"),
        bat_duoc_top100=(LABEL, lambda s: int(s[test_band.loc[s.index, "in_top100"] == 1].sum())),
    )
    .sort_values("amount_band_id")
)
band_summary["bo_sot"] = band_summary["gian_lan"] - band_summary["bat_duoc_top100"]
band_summary

Mô hình có PR-AUC cao nhất: random_forest | Trong Top 100: bắt đúng 93, báo động giả 7 | Bỏ sót 108/201

Thống kê Amount của các giao dịch gian lận bị bỏ sót:
count     108.00
mean      168.88
std       355.83
min         0.00
25%         1.00
50%         7.59
75%       106.54
max      2125.87
Name: amount, dtype: float64


,amount_band_id,amount_band,so_giao_dich,gian_lan,bat_duoc_top100,bo_sot
0,0,0,864,16,6,10
1,1,0-10,49351,88,38,50
2,2,10-50,44807,28,13,15
3,3,50-100,17631,15,11,4
4,4,100-500,22521,37,22,15
5,5,500-1000,2926,13,3,10
6,6,>1000,1390,4,0,4


## 9. So sánh hai họ mô hình: Phân lớp có giám sát (E - Mong) vs. Phát hiện bất thường (D - Huy) (Mục E7 & E8)
Thực hiện đối sánh trên cả **2 giao thức** để vừa khớp 1-1 với file `outputs/anomaly_scores_test.csv` của Huy (`split_random`), vừa có đủ điểm theo ngày (`split_temporal`) cho kho dữ liệu PostgreSQL của Quỳnh:
1. **Bảng E7.1 (`split_random` — Test 56.746 dòng, 95 gian lận):** Chạy 3 mô hình có giám sát trên đúng tập `train/val/test` của `split_random` và ghép trực tiếp theo `txn_id` với `outputs/anomaly_scores_test.csv` của Huy.
2. **Bảng E7.2 (`split_temporal` — Test Ngày 1: 139.490 dòng, 201 gian lận, Top-100/ngày):** Chạy cấu hình đã chốt của Huy (`IsolationForest(n_estimators=100, random_state=42)` và `LocalOutlierFactor(n_neighbors=100, novelty=True)` trên 20.000 mẫu train Ngày 0) để so sánh trên cùng ngân sách 100 giao dịch/ngày.

**Ghi chú bản chất hai họ mô hình (Mục E8):**
- **Phân lớp có giám sát (Supervised):** Học trực tiếp biên quyết định từ nhãn lịch sử `is_fraud`, do đó đạt `PR-AUC` và `Precision@100` cao hơn hẳn đối với các mẫu gian lận có quy luật tương tự dữ liệu huấn luyện. Tuy nhiên, trong thực tế nhãn gian lận thường đến trễ (sau khi chủ thẻ khiếu nại/tra soát) và có thể bỏ sót các phương thức tấn công mới chưa từng có nhãn.
- **Phát hiện bất thường không giám sát (Isolation Forest, LOF):** Không dùng nhãn khi huấn luyện mà chỉ đo mức độ cô lập hoặc mật độ cục bộ. Vì giao dịch hợp pháp nhưng hiếm (mua món tiền lớn, hành vi khác thường) cũng có điểm bất thường cao nên tỷ lệ dương tính giả (FP) trong Top 100 cao hơn mô hình có giám sát; đổi lại mô hình hoạt động được ngay cả khi chưa có nhãn lịch sử.

In [9]:
# --- 9.1 Đối sánh 1-1 với file anomaly_scores_test.csv của Huy trên split_random ---
tr_r = df[df["split_random"] == "train"]
va_r = df[df["split_random"] == "val"]
te_r = df[df["split_random"] == "test"]

huy_scores = pd.read_csv(OUT / "anomaly_scores_test.csv").sort_values("txn_id").reset_index(drop=True)
te_r_sorted = te_r.sort_values("txn_id").reset_index(drop=True)
assert np.array_equal(huy_scores["txn_id"].to_numpy(), te_r_sorted["txn_id"].to_numpy()), "txn_id không khớp với file của Huy!"

y_tr_r, y_va_r, y_te_r = tr_r[LABEL].to_numpy(), va_r[LABEL].to_numpy(), te_r_sorted[LABEL].to_numpy()
comp_random_rows = []

for name, cfgs in grid.items():
    best_ap, best_cfg = -1.0, None
    for cfg in cfgs:
        m = make_model(name, **cfg).fit(tr_r[FEATURES], y_tr_r)
        ap = average_precision_score(y_va_r, m.predict_proba(va_r[FEATURES])[:, 1])
        if ap > best_ap:
            best_ap, best_cfg = ap, cfg
    # Để công bằng với Huy (Huy chỉ fit trên train của split_random, không gộp val), ta fit trên train:
    m_te = make_model(name, **best_cfg).fit(tr_r[FEATURES], y_tr_r)
    s_te = m_te.predict_proba(te_r_sorted[FEATURES])[:, 1]
    r = report(name, y_te_r, s_te, K)
    comp_random_rows.append({"family": "Supervised (Mong - E)", **r, "config": json.dumps(best_cfg)})

for col_score, m_name in [("lof_score", "LOF (n_neighbors=100)"), ("isolation_forest_score", "Isolation Forest")]:
    r = report(m_name, y_te_r, huy_scores[col_score].to_numpy(), K)
    comp_random_rows.append({"family": "Unsupervised Anomaly (Huy - D)", **r, "config": "Huy notebook 03"})

rb_random = random_baseline(y_te_r, K)
comp_random_rows.append({"family": "Baseline", "model": "random_baseline_mean", **rb_random, "config": "100 runs"})

comp_random_df = pd.DataFrame(comp_random_rows).sort_values("PR_AUC", ascending=False).reset_index(drop=True)
print("=== BẢNG E7.1: SO SÁNH HAI HỌ MÔ HÌNH TRÊN CÙNG TẬP TEST split_random (56.746 dòng, 95 fraud) ===")
print(comp_random_df[["family", "model", "PR_AUC", "Precision@100", "Recall@100", "TP", "FP", "FN"]].to_string(index=False))

# --- 9.2 Chạy cấu hình Anomaly Detection của Huy trên split_temporal (Ngày 1) để so sánh theo ngày & nạp kho Quỳnh ---
sc_temp = StandardScaler()
X_d0_v = sc_temp.fit_transform(day0[V_COLS])
X_te_v = sc_temp.transform(test[V_COLS])

iso_temp = IsolationForest(n_estimators=100, contamination="auto", random_state=RANDOM_STATE, n_jobs=-1)
iso_temp.fit(day0[V_COLS])
iso_temp_scores = -iso_temp.decision_function(test[V_COLS])

rng_lof = np.random.default_rng(RANDOM_STATE)
lof_idx = rng_lof.choice(len(X_d0_v), size=min(20000, len(X_d0_v)), replace=False)
lof_temp = LocalOutlierFactor(n_neighbors=100, novelty=True, n_jobs=-1)
lof_temp.fit(X_d0_v[lof_idx])
lof_temp_scores = -lof_temp.decision_function(X_te_v)

comp_temp_rows = [
    {"family": "Supervised (Mong - E)", **r}
    for r in final_rows if r["model"] != "random_baseline_mean"
] + [
    {"family": "Unsupervised Anomaly (Huy config)", **report("LOF (n_neighbors=100)", y_te, lof_temp_scores, K)},
    {"family": "Unsupervised Anomaly (Huy config)", **report("Isolation Forest", y_te, iso_temp_scores, K)},
    {"family": "Baseline", "model": "random_baseline_mean", **rb_test},
]
comp_temp_df = pd.DataFrame(comp_temp_rows).sort_values("PR_AUC", ascending=False).reset_index(drop=True)
print("\n=== BẢNG E7.2: SO SÁNH HAI HỌ MÔ HÌNH TRÊN NGÀY 1 split_temporal (139.490 dòng, 201 fraud, K=100/ngày) ===")
comp_temp_df[["family", "model", "PR_AUC", "Precision@100", "Recall@100", "TP", "FP", "FN"]]

=== BẢNG E7.1: SO SÁNH HAI HỌ MÔ HÌNH TRÊN CÙNG TẬP TEST split_random (56.746 dòng, 95 fraud) ===
                        family                 model   PR_AUC  Precision@100  Recall@100    TP    FP    FN
         Supervised (Mong - E)         random_forest 0.809683         0.7700    0.810526 77.00 23.00 18.00
         Supervised (Mong - E)               hist_gb 0.768332         0.7200    0.757895 72.00 28.00 23.00
         Supervised (Mong - E)                logreg 0.677267         0.7400    0.778947 74.00 26.00 21.00
Unsupervised Anomaly (Huy - D) LOF (n_neighbors=100) 0.312462         0.3800    0.400000 38.00 62.00 57.00
Unsupervised Anomaly (Huy - D)      Isolation Forest 0.160945         0.2700    0.284211 27.00 73.00 68.00
                      Baseline  random_baseline_mean 0.001872         0.0026    0.002737  0.26 99.74 94.74

=== BẢNG E7.2: SO SÁNH HAI HỌ MÔ HÌNH TRÊN NGÀY 1 split_temporal (139.490 dòng, 201 fraud, K=100/ngày) ===


,family,model,PR_AUC,Precision@100,Recall@100,TP,FP,FN
0,Supervised (Mong - E),random_forest,0.784326,0.9300,0.462687,93.00,7.00,108.00
1,Supervised (Mong - E),logreg,0.723883,0.9400,0.467662,94.00,6.00,107.00
2,Supervised (Mong - E),hist_gb,0.672652,0.9500,0.472637,95.00,5.00,106.00
3,Unsupervised Anomaly (Huy config),LOF (n_neighbors=100),0.247834,0.3000,0.149254,30.00,70.00,171.00
4,Unsupervised Anomaly (Huy config),Isolation Forest,0.158203,0.3100,0.154229,31.00,69.00,170.00
5,Baseline,random_baseline_mean,0.001541,0.0015,0.000746,0.15,99.85,200.85


## 10. Lưu kết quả cho Báo cáo, Dashboard Superset và Kho PostgreSQL của Quỳnh (Mục E9)
- `outputs/classification_results.csv` và `outputs/classification_validation.csv`: bảng kết quả chính thức của Phần E trên `split_temporal`.
- `outputs/model_family_comparison_random_test.csv` và `outputs/model_family_comparison_temporal_test.csv`: bảng đối sánh E7 giữa Supervised (Mong) và Anomaly Detection (Huy).
- `outputs/classification_scores_test.csv` (chỉ lưu local, không commit): có đủ `txn_id` (1-based) và `row_id` (0-based), `Time`, `Amount`, `amount_band`, `Class` và điểm 3 mô hình.
- `outputs/model_scores_warehouse.csv`: đúng định dạng 6 cột (`txn_id,split_set,iforest_score,lof_score,sup_prob_score,model_run_note`) để script `sql/05-load-model-scores.sql` của Quỳnh (Phần C) nạp vào `Fact_Transaction`.

In [10]:
# 1. Lưu bảng kết quả phân lớp và đối sánh hai họ mô hình
final[["model", "PR_AUC", "Precision@100", "Recall@100"]].to_csv(OUT / "classification_results.csv", index=False)
val_table[["model", "PR_AUC", f"Precision@{K_VAL}", f"Recall@{K_VAL}", "config"]].to_csv(OUT / "classification_validation.csv", index=False)
comp_random_df.to_csv(OUT / "model_family_comparison_random_test.csv", index=False)
comp_temp_df.to_csv(OUT / "model_family_comparison_temporal_test.csv", index=False)

# 2. Lưu classification_scores_test.csv (tương thích cả Superset CSV lẫn txn_id chuẩn nhóm)
scores_df = pd.DataFrame({
    "txn_id": test["txn_id"].to_numpy(),
    "row_id": test["txn_id"].to_numpy() - 1,
    "Time": test["time_sec"].to_numpy(),
    "Amount": test["amount"].to_numpy(),
    "amount_band": test["amount_band"].to_numpy(),
    "Class": y_te,
})
for k_name, v_score in test_scores.items():
    scores_df[f"score_{k_name}"] = v_score
scores_df.to_csv(OUT / "classification_scores_test.csv", index=False)

# 3. Lưu model_scores_warehouse.csv đúng chuẩn 6 cột của Quỳnh (sql/05-load-model-scores.sql)
warehouse_scores = pd.DataFrame({
    "txn_id": test["txn_id"].to_numpy(),
    "split_set": "TEST",
    "iforest_score": np.round(iso_temp_scores, 6),
    "lof_score": np.round(lof_temp_scores, 6),
    "sup_prob_score": np.round(test_scores[top_model], 6),
    "model_run_note": f"temporal_day1_dedup_sup={top_model}",
})
warehouse_scores.to_csv(OUT / "model_scores_warehouse.csv", index=False, lineterminator="\n")

print("Đã xuất các file:", [
    "classification_results.csv",
    "classification_validation.csv",
    "model_family_comparison_random_test.csv",
    "model_family_comparison_temporal_test.csv",
    "classification_scores_test.csv",
    "model_scores_warehouse.csv",
])

Đã xuất các file: ['classification_results.csv', 'classification_validation.csv', 'model_family_comparison_random_test.csv', 'model_family_comparison_temporal_test.csv', 'classification_scores_test.csv', 'model_scores_warehouse.csv']


## 11. Hạn chế và lưu ý diễn giải (Mục E8 & Tiêu chí 10)
- Tập Test Ngày 1 (sau khử trùng `DQ-01`) có `139.490` giao dịch và `201` giao dịch gian lận; chênh lệch 1–2 giao dịch trong Top 100 (`94` vs. `95` ca giữa Random Forest, Logistic Regression và HistGradientBoosting) là rất nhỏ, không dùng để kết luận tuyệt đối mô hình nào vượt trội về mặt thống kê.
- Với ngân sách rà soát `100` giao dịch/ngày, trần lý thuyết `Recall@100` ở Ngày 1 là `100 / 201 = 49,75%`. Cả 3 mô hình có giám sát bắt được `94–95` giao dịch gian lận (`Recall@100` đạt `46,8% – 47,3%`), tức đã đạt ~`94% – 95%` mức trần lý thuyết mà ngân sách cho phép.
- `V1..V28` là các thành phần chính PCA đã ẩn danh, không diễn giải ý nghĩa nghiệp vụ cho từng cột `V*`.
- Khai báo sử dụng AI: được ghi đầy đủ trong `README_E_Mong.md` và báo cáo chung của nhóm.